In [4]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings, os, joblib
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

np.random.seed(42)

# Climate zone → integer code
ZONE_MAP = {
    "tropical":    0,
    "subtropical": 1,
    "semiarid":    2,
    "coastal":     3,
    "highland":    4,
}

# Soil type → integer code
SOIL_MAP = {
    "loamy":    0,
    "clay":     1,
    "sandy":    2,
    "laterite": 3,
    "alluvial": 4,
    "black":    5,
    "red":      6,
}

# Irrigation availability → integer code
IRR_MAP = {
    "yes":     0,
    "partial": 1,
    "no":      2,
}

# Geographic region → integer code
REGION_MAP = {
    "india_south": 0,
    "india_ne":    1,
    "india_west":  2,
    "india_north": 3,
    "sea":         4,
    "pacific":     5,
    "other":       6,
}

# Feature column names
FEATURES = [
    "zone", "soil", "rainfall_mm", "temp_c", "irrigation",
    "region", "humidity_pct", "elevation_m", "ph", "sunlight_hrs"
]

CROP_CATALOGUE = {
    "Betel Palm (Areca)": {"ideal_zone": [0, 1, 3], "ideal_soil": [0, 3, 4], "ideal_region": [0, 1, 4], "min_rainfall": 750, "max_temp": 38, "min_temp": 14, "drought_tolerance": 1, "water_need": "Medium", "profitability": 3, "season": "Early monsoon (May–Aug)", "description": "Primary plantation crop"},
    "Coconut": {"ideal_zone": [0, 3], "ideal_soil": [0, 4, 2], "ideal_region": [0, 4, 5], "min_rainfall": 1000, "max_temp": 40, "min_temp": 15, "drought_tolerance": 1, "water_need": "Medium–High", "profitability": 3, "season": "Early monsoon (May–Aug)", "description": "Iconic coastal crop"},
    "Banana": {"ideal_zone": [0, 1], "ideal_soil": [0, 4, 5], "ideal_region": [0, 1, 4], "min_rainfall": 1200, "max_temp": 38, "min_temp": 15, "drought_tolerance": 0, "water_need": "High", "profitability": 2, "season": "Pre-monsoon (Feb–Apr)", "description": "Fast-growing"},
    "Black Pepper": {"ideal_zone": [0, 3], "ideal_soil": [0, 3, 4], "ideal_region": [0, 4], "min_rainfall": 1500, "max_temp": 35, "min_temp": 10, "drought_tolerance": 0, "water_need": "High", "profitability": 3, "season": "Early monsoon (May–Jun)", "description": "High value spice"},
    "Coffee (Arabica)": {"ideal_zone": [4, 0], "ideal_soil": [0, 3], "ideal_region": [0], "min_rainfall": 1600, "max_temp": 28, "min_temp": 13, "drought_tolerance": 0, "water_need": "High", "profitability": 3, "season": "Post-monsoon (Oct–Nov)", "description": "Cool highland crop"},
    "Coffee (Robusta)": {"ideal_zone": [0, 4], "ideal_soil": [0, 3, 4], "ideal_region": [0], "min_rainfall": 1400, "max_temp": 32, "min_temp": 12, "drought_tolerance": 1, "water_need": "Medium–High", "profitability": 3, "season": "Post-monsoon (Oct–Nov)", "description": "Hardier than Arabica"},
    "Rubber": {"ideal_zone": [0], "ideal_soil": [0, 3, 4], "ideal_region": [0, 4], "min_rainfall": 1800, "max_temp": 35, "min_temp": 20, "drought_tolerance": 0, "water_need": "Very High", "profitability": 3, "season": "Early monsoon (Jun–Jul)", "description": "Tropical perennial"},
    "Turmeric": {"ideal_zone": [0, 1], "ideal_soil": [0, 4, 5], "ideal_region": [0, 1], "min_rainfall": 1000, "max_temp": 35, "min_temp": 18, "drought_tolerance": 1, "water_need": "Medium", "profitability": 2, "season": "Early monsoon (May–Jun)", "description": "Rhizome spice"},
    "Ginger": {"ideal_zone": [0, 4], "ideal_soil": [0, 4], "ideal_region": [0, 1], "min_rainfall": 1200, "max_temp": 30, "min_temp": 18, "drought_tolerance": 0, "water_need": "High", "profitability": 3, "season": "Early monsoon (Apr–May)", "description": "High-value spice"},
    "Cardamom": {"ideal_zone": [4, 0], "ideal_soil": [0, 3], "ideal_region": [0], "min_rainfall": 1500, "max_temp": 30, "min_temp": 10, "drought_tolerance": 0, "water_need": "High", "profitability": 3, "season": "Pre-monsoon (Apr–May)", "description": "Queen of spices"},
    "Sugarcane": {"ideal_zone": [0, 1, 2], "ideal_soil": [0, 4, 5], "ideal_region": [0, 2, 3], "min_rainfall": 750, "max_temp": 40, "min_temp": 20, "drought_tolerance": 1, "water_need": "Medium–High", "profitability": 2, "season": "Post-monsoon (Oct–Dec)", "description": "Bulk cash crop"},
    "Cotton": {"ideal_zone": [2, 1], "ideal_soil": [5, 0, 6], "ideal_region": [0, 2], "min_rainfall": 500, "max_temp": 40, "min_temp": 20, "drought_tolerance": 2, "water_need": "Low–Medium", "profitability": 2, "season": "Pre-monsoon (Apr–Jun)", "description": "Semi-arid crop"},
    "Groundnut": {"ideal_zone": [2, 1], "ideal_soil": [2, 6, 0], "ideal_region": [0, 2], "min_rainfall": 500, "max_temp": 38, "min_temp": 22, "drought_tolerance": 2, "water_need": "Low", "profitability": 2, "season": "Early monsoon (Jun–Jul)", "description": "Drought-tolerant legume"},
    "Rice (Paddy)": {"ideal_zone": [0, 1], "ideal_soil": [1, 4, 5], "ideal_region": [0, 1, 4], "min_rainfall": 1200, "max_temp": 37, "min_temp": 22, "drought_tolerance": 0, "water_need": "Very High", "profitability": 1, "season": "Early monsoon (Jun–Jul)", "description": "Staple crop"},
    "Maize (Corn)": {"ideal_zone": [1, 2, 0], "ideal_soil": [0, 4, 6], "ideal_region": [0, 3], "min_rainfall": 500, "max_temp": 38, "min_temp": 18, "drought_tolerance": 1, "water_need": "Medium", "profitability": 1, "season": "Early monsoon (Jun–Jul)", "description": "Versatile cereal"},
    "Tapioca (Cassava)": {"ideal_zone": [0, 2], "ideal_soil": [2, 6, 0], "ideal_region": [0, 4], "min_rainfall": 500, "max_temp": 40, "min_temp": 18, "drought_tolerance": 2, "water_need": "Low", "profitability": 1, "season": "Post-monsoon (Oct–Nov)", "description": "Drought-tolerant starch"},
    "Tomato": {"ideal_zone": [1, 2, 0], "ideal_soil": [0, 4, 6], "ideal_region": [0, 2], "min_rainfall": 600, "max_temp": 35, "min_temp": 18, "drought_tolerance": 1, "water_need": "Medium", "profitability": 2, "season": "Post-monsoon (Sep–Oct)", "description": "High-demand vegetable"},
    "Onion": {"ideal_zone": [1, 2], "ideal_soil": [0, 6, 4], "ideal_region": [0, 2], "min_rainfall": 500, "max_temp": 35, "min_temp": 15, "drought_tolerance": 1, "water_need": "Low–Medium", "profitability": 2, "season": "Post-monsoon (Oct–Nov)", "description": "Short duration vegetable"},
    "Jackfruit": {"ideal_zone": [0, 1], "ideal_soil": [0, 4, 3], "ideal_region": [0, 4], "min_rainfall": 1000, "max_temp": 38, "min_temp": 16, "drought_tolerance": 1, "water_need": "Medium", "profitability": 2, "season": "Post-monsoon (Oct–Dec)", "description": "Large canopy tree"},
    "Mango": {"ideal_zone": [1, 2, 0], "ideal_soil": [0, 6, 4], "ideal_region": [0, 2, 3], "min_rainfall": 500, "max_temp": 40, "min_temp": 15, "drought_tolerance": 2, "water_need": "Low–Medium", "profitability": 3, "season": "Post-monsoon (Oct–Nov)", "description": "King of fruits"}
}

ALL_CROPS = sorted(CROP_CATALOGUE.keys())
CROP_INDEX_TO_NAME = {i: name for i, name in enumerate(ALL_CROPS)}
CROP_NAME_TO_INDEX = {name: i for i, name in enumerate(ALL_CROPS)}
SEASON_LABELS = ["Early monsoon (May–Aug)", "Pre-monsoon (Feb–Apr)", "Post-monsoon (Oct–Dec)", "Pre-monsoon (Apr–Jun)"]
DROUGHT_LABELS = ["Low risk", "Medium risk", "High risk"]

def _score_crop(crop_name, zone, soil, rainfall, temp, irrigation, region, humidity, elevation, ph, sunlight):
    crop = CROP_CATALOGUE[crop_name]
    score = 0
    if zone in crop["ideal_zone"]: score += 25
    if soil in crop["ideal_soil"]: score += 20
    if region in crop["ideal_region"]: score += 15
    rain_ratio = min(rainfall / max(crop["min_rainfall"], 1), 2.0)
    if rainfall >= crop["min_rainfall"]: score += min(20, int(rain_ratio * 10))
    else: score -= int(((crop["min_rainfall"] - rainfall) / crop["min_rainfall"]) * 15)
    if crop["min_temp"] <= temp <= crop["max_temp"]: score += 10
    if irrigation == 2 and crop["drought_tolerance"] == 2: score += 8
    elif irrigation == 0 and crop["drought_tolerance"] == 0: score += 3
    if 5.5 <= ph <= 7.0: score += 4
    return max(0, min(100, score))

def _best_crop_label(*args):
    scores = {name: _score_crop(name, *args) for name in ALL_CROPS}
    return CROP_NAME_TO_INDEX[max(scores, key=scores.get)]

def _season_label(zone, rainfall, temp):
    if zone == 4: return 2
    elif zone in (0, 1) and rainfall > 1200: return 0
    elif zone == 2 or temp > 32: return 3
    else: return 1

def _drought_label(zone, soil, rainfall, irrigation, temp, humidity):
    score = (2 if soil == 2 else 0) + (2 if irrigation == 2 else 0) + (2 if zone == 2 else 0) + (3 if rainfall < 700 else 1 if rainfall < 1000 else 0)
    return 0 if score <= 2 else (1 if score <= 5 else 2)

def _companion_crop_label(zone, soil, region, rainfall, irrigation, best_crop_idx):
    primary_name = CROP_INDEX_TO_NAME[best_crop_idx]
    companion_scores = {name: 0 for name in ALL_CROPS if name != primary_name}
    for name in companion_scores:
        crop = CROP_CATALOGUE[name]
        if zone in crop["ideal_zone"]: companion_scores[name] += 10
        if crop["water_need"] in ("Low", "Low–Medium"): companion_scores[name] += 5
    return CROP_NAME_TO_INDEX[max(companion_scores, key=companion_scores.get)]

def generate_dataset(n=7000):
    rows = []
    for _ in range(n):
        zone, soil, region = np.random.randint(0, 5), np.random.randint(0, 7), np.random.randint(0, 7)
        rainfall = int(np.clip(np.random.normal(1300, 600), 400, 3800))
        temp = int(np.clip(np.random.normal(28, 5), 15, 42))
        irrigation, humidity = np.random.randint(0, 3), int(np.clip(np.random.normal(65, 15), 30, 98))
        elevation, ph = int(np.clip(np.random.normal(200, 200), 0, 1200)), round(np.clip(np.random.normal(6.3, 0.8), 4.0, 8.5), 1)
        sunlight = round(np.clip(np.random.normal(7.5, 1.5), 4.0, 12.0), 1)
        best = _best_crop_label(zone, soil, rainfall, temp, irrigation, region, humidity, elevation, ph, sunlight)
        rows.append([zone, soil, rainfall, temp, irrigation, region, humidity, elevation, ph, sunlight, best, _season_label(zone, rainfall, temp), _drought_label(zone, soil, rainfall, irrigation, temp, humidity), _companion_crop_label(zone, soil, region, rainfall, irrigation, best)])
    return pd.DataFrame(rows, columns=FEATURES + ["best_crop", "season_label", "drought_label", "companion_crop"])

def train_models(df):
    X = df[FEATURES].values
    results = {}
    configs = [("best_crop", ALL_CROPS), ("season_label", SEASON_LABELS), ("drought_label", DROUGHT_LABELS), ("companion_crop", ALL_CROPS)]
    for target, labels in configs:
        y = df[target].values
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
        model = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42).fit(X_tr, y_tr)
        results[target] = {"model": model, "acc": accuracy_score(y_te, model.predict(X_te)), "labels": labels}
    return results

def plot_results(results, df, out="/content/crop_advisor_report.png"):
    plt.figure(figsize=(10, 6))
    plt.bar(results.keys(), [results[k]["acc"] for k in results])
    plt.title("Model Accuracy")
    plt.savefig(out)
    print(f"Chart saved -> {out}")

def save_models(results, path="/content/models"):
    os.makedirs(path, exist_ok=True)
    for key in results: joblib.dump(results[key]["model"], os.path.join(path, f"{key}.joblib"))
    print(f"Models saved -> {path}")

def main():
    df = generate_dataset(7000)
    results = train_models(df)
    plot_results(results, df)
    save_models(results)
    return results

if __name__ == "__main__":
    main()

Chart saved -> /content/crop_advisor_report.png
Models saved -> /content/models


In [5]:
results = main()   # train once

advise(results,
    zone="semiarid", soil="laterite", rainfall_mm=850, temp_c=29,
    irrigation="partial", region="india_south", humidity_pct=62,
    elevation_m=380, ph=6.0, sunlight_hrs=8.5,
    location_name="My Farm — Dharmapuri")

Chart saved -> /content/crop_advisor_report.png
Models saved -> /content/models

══════════════════════════════════════════════════════════════
  CROP ADVISORY REPORT
  Location : My Farm — Dharmapuri
  Inputs   : SEMIARID zone | laterite soil | 850mm rain | 29°C | partial irrigation
══════════════════════════════════════════════════════════════

  FARM CONDITIONS SUMMARY
  Zone                  : Semiarid (elevation 380m)
  Soil type             : Laterite
  Annual rainfall       : 850 mm/year
  Temperature           : 29°C average
  Irrigation            : Partial
  Humidity              : 62%
  Soil pH               : 6.0
  Daily sunlight        : 8.5 hours

  RECOMMENDED PLANTING SEASON
  Best window           : Pre-monsoon (Apr–Jun)
  Confidence            : 94.0%

  DROUGHT / DRYING RISK
  Risk level            : Medium risk [MED]
  Confidence            : 85.0%
  ~  ACTION: Prepare water retention basins, consider mulching

  ML-RECOMMENDED BEST CROP (top 3)
  1. Coffee (Arabica

{'ranked_crops': [('Cotton', 71),
  ('Groundnut', 71),
  ('Maize (Corn)', 71),
  ('Mango', 71),
  ('Onion', 71),
  ('Tapioca (Cassava)', 71),
  ('Tomato', 68),
  ('Sugarcane', 65),
  ('Betel Palm (Areca)', 60),
  ('Jackfruit', 47),
  ('Coffee (Robusta)', 44),
  ('Black Pepper', 43),
  ('Cardamom', 43),
  ('Rubber', 42),
  ('Coffee (Arabica)', 32),
  ('Coconut', 27),
  ('Turmeric', 27),
  ('Banana', 25),
  ('Ginger', 25),
  ('Rice (Paddy)', 25)],
 'model_outputs': {'best_crop': {'prediction': 'Cotton',
   'confidence': np.float64(52.0),
   'top3': [('Coffee (Arabica)', np.float64(52.0)),
    ('Betel Palm (Areca)', np.float64(39.0)),
    ('Ginger', np.float64(7.0))],
   'probabilities': {'Banana': np.float64(0.0),
    'Betel Palm (Areca)': np.float64(39.0),
    'Black Pepper': np.float64(0.0),
    'Cardamom': np.float64(0.0),
    'Coconut': np.float64(0.0),
    'Coffee (Arabica)': np.float64(52.0),
    'Coffee (Robusta)': np.float64(0.0),
    'Cotton': np.float64(1.0),
    'Ginger': np.f